# 🌿 CropGuard NE — Model Training on GPU

**Runtime → Change runtime type → GPU (T4)**

Expected training time: ~25–40 minutes on T4 GPU

In [ ]:
# 1. Mount Google Drive (to save model)
from google.colab import drive
drive.mount('/content/drive')

In [2]:
# 2. Install Kaggle and upload credentials
!pip install kaggle -q
from google.colab import files
print('Upload your kaggle.json from https://kaggle.com/settings → Create New Token')
files.upload()  # upload kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# 3. Download PlantVillage dataset
!kaggle datasets download abdallahalidev/plantvillage-dataset
!unzip -q plantvillage-dataset.zip -d data/
import os
color_dir = 'data/plantvillage dataset/color'
classes = os.listdir(color_dir)
print(f'Classes: {len(classes)}')

In [ ]:
# 4. Install dependencies
!pip install -q tensorflow pillow numpy scikit-learn matplotlib seaborn

In [ ]:
import os, json, numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

DATA_DIR  = 'data/plantvillage dataset/color'
MODEL_DIR = '/content/drive/MyDrive/CropGuardNE'
os.makedirs(MODEL_DIR, exist_ok=True)

IMG_SIZE   = (224, 224)
BATCH_SIZE = 64   # larger batch for GPU
SEED       = 42

train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=20, width_shift_range=0.15,
    height_shift_range=0.15, shear_range=0.1, zoom_range=0.2,
    horizontal_flip=True, brightness_range=[0.8,1.2],
    fill_mode='nearest', validation_split=0.2
)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = train_datagen.flow_from_directory(DATA_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', subset='training', seed=SEED)
val_gen   = val_datagen.flow_from_directory(DATA_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', subset='validation', seed=SEED)

NUM_CLASSES = len(train_gen.class_indices)
class_indices = {v: k for k, v in train_gen.class_indices.items()}
with open(f'{MODEL_DIR}/class_indices.json', 'w') as f:
    json.dump(class_indices, f)
print(f'Classes: {NUM_CLASSES}, Train: {train_gen.samples}, Val: {val_gen.samples}')

In [ ]:
# Build model
base = MobileNetV2(input_shape=(*IMG_SIZE,3), include_top=False, weights='imagenet')
base.trainable = False

inp = tf.keras.Input(shape=(*IMG_SIZE,3))
x = base(inp, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = models.Model(inp, out)

# Phase 1
model.compile(optimizer=optimizers.Adam(1e-4), loss='categorical_crossentropy',
              metrics=['accuracy'])
cb1 = [callbacks.EarlyStopping('val_accuracy', patience=5, restore_best_weights=True),
       callbacks.ModelCheckpoint(f'{MODEL_DIR}/phase1_best.h5', save_best_only=True)]
model.fit(train_gen, validation_data=val_gen, epochs=10, callbacks=cb1)

In [ ]:
# Phase 2 — fine-tune
base.trainable = True
for layer in base.layers[:-50]: layer.trainable = False

model.compile(optimizer=optimizers.Adam(1e-5), loss='categorical_crossentropy',
              metrics=['accuracy'])
cb2 = [callbacks.EarlyStopping('val_accuracy', patience=7, restore_best_weights=True),
       callbacks.ModelCheckpoint(f'{MODEL_DIR}/plantdisease_mobilenetv2.h5', save_best_only=True),
       callbacks.ReduceLROnPlateau('val_loss', factor=0.3, patience=4)]
hist = model.fit(train_gen, validation_data=val_gen, epochs=20, callbacks=cb2)

best = max(hist.history['val_accuracy'])
print(f'\n✅ Best val accuracy: {best*100:.2f}%')
print(f'Model saved to: {MODEL_DIR}/plantdisease_mobilenetv2.h5')

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(hist.history['accuracy'], label='Train', color='#1D9E75')
plt.plot(hist.history['val_accuracy'], label='Val', color='#D85A30')
plt.title('Accuracy'); plt.legend(); plt.grid(alpha=0.3)
plt.subplot(1,2,2)
plt.plot(hist.history['loss'], label='Train', color='#1D9E75')
plt.plot(hist.history['val_loss'], label='Val', color='#D85A30')
plt.title('Loss'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/training_curves.png', dpi=150)
plt.show()